In [ ]:
import mcstasscript as ms
import mcstastox as mx
import scipp as sc
from scipp.typing import VariableLike
import scipp as sc
from scippneutron.conversion.graph.beamline import beamline
import os
import sys
import numpy as np

parent = os.path.dirname(os.getcwd())

sys.path.append(parent)
from trex_reduction import inelastic
from trex_reduction import produce_trex_event_object


In [ ]:

file_path = parent + "/runs/LET_vanad"

with mx.Read(file_path) as loaded_data:
    scipp_object = loaded_data.export_scipp(source_name="SourceMantid",
                                            sample_name="iso_samp")
    
data = ms.load_data(file_path)

In [ ]:
# Load event data into scipp 
event_object = scipp_object
# McStas provides absolute time, not time of flight
event_object['events'].bins.coords['tof'] = event_object['events'].bins.coords['t']
# Add additional information required for inelastic scattering
# test = produce_trex_event_object(event_object['events'].bins, file_path, "Monitor6")
# event_object

In [ ]:
event_object

# Calculate minimum and maximum kf

In [ ]:
from scipy.ndimage import label

Lm =  23.505

def _calc_kf_from_ef(ef):
    hbar = sc.constants.hbar
    m_n = sc.constants.neutron_mass

    kf = sc.sqrt(2 * m_n * ef) / hbar

    return kf.to(unit='1/angstrom')

def _calc_pulse_tof_centroid(tof_monitor,threshold=0, to_s=1e-6):
    # Assumes all pulses are evenly spaced in xaxis - but isn't bin data??? confirm this
    # Confirm correct way to axis tof data - is xaxis correct????? 
    mask = tof_monitor.Intensity != threshold
    labels, num_features = label(mask)
    weighted_sum = np.bincount(labels, weights=tof_monitor.xaxis * tof_monitor.Intensity)[1:]
    weight_total = np.bincount(labels, weights=tof_monitor.Intensity)[1:]

    coms = weighted_sum / weight_total * to_s
    return coms

monitor = ms.name_search("Monitor6", data)
tom_centroid = _calc_pulse_tof_centroid(monitor)

vi = Lm / tom_centroid[0] * sc.Unit('m/s')

ei = (0.5 * sc.constants.m_n * vi**2).to(unit='meV')

unit_ki = sc.vector([0, 0, 1])
mag_ki = (sc.constants.neutron_mass * vi) / sc.constants.hbar

ki = unit_ki * mag_ki

prop_ei = 0.8
max_ef = (1+prop_ei) * ei
min_ef = (1-prop_ei) * ei

min_kf = _calc_kf_from_ef(min_ef)
max_kf = _calc_kf_from_ef(max_ef)

print(f'min_kf: {min_kf}, max_kf: {max_kf}')


# Access and calculate detector pixel Qs

In [52]:
sample_position = sc.vector([0, 0, 25], unit='m')



pixel_kf = event_object['positions'] - sample_position
pixel_unit_kf = pixel_kf / sc.norm(pixel_kf)
pixel_Q_vector = unit_ki - pixel_unit_kf

pixel_Q_vector

<scipp.Variable> (pixel_id: 30056)    vector3  [dimensionless]  [(0.554355, 0.493381, 0.329727), (0.544771, 0.493381, 0.321915), ..., (-0.573185, -0.493381, 1.65424), (-0.563827, -0.493381, 1.66232)]